<a href="https://colab.research.google.com/github/VladAgapov1969/VMAR_multi_agent_demos/blob/main/magent_v4_Rag_Qwen_Qwen_RL_v1_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install chromadb sentence-transformers numpy requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 140.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Foun

In [3]:
# ============================================================
# 🚀 RAG MULTI-AGENT SYSTEM (COLAB OPTIMIZED VERSION)
# ============================================================
# Эта версия использует Hugging Face Transformers вместо Ollama,
# что позволяет работать в Colab без локального сервера.
# ============================================================

import requests
import json
import time
import re
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np
from chromadb.api.types import EmbeddingFunction
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ============================================================
# ЗАГРУЗКА МОДЕЛИ ИЗ HUGGING FACE (ВМЕСТО OLLAMA)
# ============================================================

print("🔄 Загрузка модели Qwen из Hugging Face...")
MODEL_NAME = "Qwen/Qwen2.5-1.5B"  # Или "Qwen/Qwen2.5-0.5B" для более быстрой загрузки

# Загрузка токенизатора и модели
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,  # Экономия памяти
    device_map="auto",  # Автоматическое распределение на GPU/CPU
    trust_remote_code=True
)
print("✅ Модель загружена!")

def generate_response(prompt, max_new_tokens=512, temperature=0.7):
    """
    Генерация ответа через модель Hugging Face.
    Заменяет Ollama.generate().
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Убираем промпт из ответа (чтобы получить только сгенерированную часть)
    response = response[len(prompt):].strip()
    return response

# ============================================================
# RAG SYSTEM (БЕЗ ИЗМЕНЕНИЙ)
# ============================================================

class RAGSystem:
    """Simple RAG system with a knowledge base"""

    def __init__(self, knowledge_base=None):
        print("🔄 Loading embedding model (all-MiniLM-L6-v2)...")
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        print("✅ Embedding model loaded!")

        self.chroma_client = chromadb.PersistentClient(path="./rag_db")

        class CustomEmbeddingFunction(EmbeddingFunction):
            def __init__(self, model):
                self.model = model
                self._name = "custom_embedding"

            def __call__(self, input):
                if isinstance(input, str):
                    input = [input]
                embeddings = self.model.encode(input)
                if isinstance(embeddings, np.ndarray):
                    if embeddings.ndim == 1:
                        return [embeddings.tolist()]
                    return embeddings.tolist()
                return embeddings

            def name(self):
                return self._name

        embedding_func = CustomEmbeddingFunction(self.embedding_model)

        try:
            existing_collections = self.chroma_client.list_collections()
            if "knowledge_base" in [col.name for col in existing_collections]:
                self.chroma_client.delete_collection("knowledge_base")
                print("🔄 Removed existing knowledge_base collection")
        except:
            pass

        self.collection = self.chroma_client.create_collection(
            name="knowledge_base",
            embedding_function=embedding_func
        )
        print("✅ Created new knowledge_base collection")

        if knowledge_base:
            self.add_knowledge(knowledge_base)

    def add_knowledge(self, documents):
        if not documents:
            return

        batch_size = 10
        for i in range(0, len(documents), batch_size):
            batch = documents[i:i+batch_size]
            ids = [f"doc_{i + j}" for j in range(len(batch))]
            try:
                self.collection.add(documents=batch, ids=ids)
            except Exception as e:
                print(f"⚠️ Error adding batch: {e}")
        print(f"✅ Added {len(documents)} documents to RAG knowledge base")

    def retrieve(self, query, n_results=3):
        if not query:
            return []

        try:
            results = self.collection.query(
                query_texts=[query],
                n_results=min(n_results, 5)
            )
            if results and results['documents'] and len(results['documents']) > 0:
                return results['documents'][0]
            return []
        except Exception as e:
            print(f"⚠️ RAG retrieval error: {e}")
            return []

# ============================================================
# MAIN MULTI-AGENT SYSTEM (АДАПТИРОВАН ДЛЯ HUGGING FACE)
# ============================================================

class RAGMultiAgentSystem:
    def __init__(self):
        # ✅ ИСПОЛЬЗУЕМ HUGGING FACE МОДЕЛЬ
        self.planner = "HF"
        self.coder = "HF"

        knowledge_base = [
            "Security Rule: All functions must handle user input validation",
            "Security Rule: Never use eval() or exec() on user input",
            "Performance Rule: Avoid recursion for large datasets (use iteration)",
            "Code Style Rule: Include docstrings for all functions - use '''docstring'''",
            "Code Style Rule: Use snake_case for function names",
        ]

        self.rag = RAGSystem(knowledge_base)
        print("🚀 Air-gapped multi-agent system with RAG ready!")
        print(f"📚 Knowledge base contains {len(knowledge_base)} rules")
        print("✅ Using Qwen from Hugging Face for both planning and coding")

    def generate(self, prompt, max_new_tokens=800, temperature=0.7):
        """Генерация через Hugging Face модель (замена Ollama.generate())"""
        return generate_response(prompt, max_new_tokens, temperature)

    def plan(self, task):
        print("🧠 Qwen is planning...")
        relevant_rules = self.rag.retrieve(task)
        rag_context = "\n".join([f"- {rule}" for rule in relevant_rules]) if relevant_rules else "No specific rules found."

        prompt = f"""Create a detailed, step-by-step plan for this task:

TASK: {task}

IMPORTANT CONTEXT/RULES FROM KNOWLEDGE BASE:
{rag_context}

Provide the plan as clear, numbered steps. Focus on the algorithm and logic.
Be specific about what the code should do.
Consider the rules above when designing your plan."""

        return self.generate(prompt, max_new_tokens=800, temperature=0.7)

    def code(self, plan):
        print("💻 Qwen is coding...")

        relevant_rules = self.rag.retrieve(plan)
        rag_context = "\n".join([f"- {rule}" for rule in relevant_rules]) if relevant_rules else "No specific rules found."

        prompt = f"""Write Python code for this task:

{plan}

Rules to follow:
{rag_context}

Important:
- Write clean, working Python code
- Include docstrings with '''docstring'''
- Use snake_case for function names
- Handle input validation
- Just write the code, no explanations

Code:"""

        response = self.generate(prompt, max_new_tokens=1500, temperature=0.3)
        return self.extract_code(response)

    def extract_code(self, text):
        if not text or text.startswith("ERROR"):
            return ""

        if "```python" in text:
            code = text.split("```python")[1].split("```")[0]
            if code.strip():
                return code.strip()

        if "```" in text:
            code = text.split("```")[1].split("```")[0]
            if code.strip():
                return code.strip()

        lines = text.split('\n')
        code_lines = []
        in_code = False

        for line in lines:
            stripped = line.strip()
            if any(stripped.startswith(kw) for kw in
                   ['def ', 'import ', 'from ', 'class ', 'print(', 'return ',
                    'if ', 'for ', 'while ', 'try:', 'except:', 'with ',
                    'else:', 'elif ', 'break', 'continue', 'pass',
                    '#', '"""', "'''"]):
                in_code = True
                code_lines.append(line)
            elif in_code and stripped:
                code_lines.append(line)
            elif in_code and not stripped:
                code_lines.append(line)
            elif in_code and stripped and not any(stripped.startswith(kw) for kw in
                  ['def ', 'import ', 'from ', 'class ', 'print(', 'return ',
                   'if ', 'for ', 'while ', 'try:', 'except:', 'with ',
                   'else:', 'elif ', 'break', 'continue', 'pass',
                   '#', '"""', "'''"]):
                break

        if code_lines:
            return '\n'.join(code_lines)

        return ""

    def process(self, task):
        """Full multi-agent workflow with RAG"""
        print(f"\n📋 Task: {task}")
        print("-" * 50)

        plan = self.plan(task)
        print(f"\n📝 Plan:\n{plan}")
        print("-" * 50)

        code = self.code(plan)
        print(f"\n💻 Code:\n{code}")
        print("-" * 50)

        print("\n⚙️ Executing code...")
        if code.strip():
            try:
                safe_globals = {
                    "__builtins__": __builtins__,
                    "print": print,
                    "range": range,
                    "int": int,
                    "float": float,
                    "str": str,
                    "list": list,
                    "dict": dict,
                }
                exec(code, safe_globals)
                print("✅ Code executed successfully!")

                func_names = ['factorial', 'calculate_factorial']
                for func_name in func_names:
                    if func_name in safe_globals:
                        test_result = safe_globals[func_name](5)
                        print(f"🧪 {func_name}(5) = {test_result} (expected: 120)")
                        break
            except Exception as e:
                print(f"❌ Code execution failed: {e}")
        else:
            print("⚠️ No code generated.")

        return {"plan": plan, "code": code}

# ============================================================
# ДЕМОНСТРАЦИЯ
# ============================================================

print("\n" + "=" * 60)
print("🚀 RUNNING MULTI-AGENT SYSTEM WITH RAG (COLAB VERSION)")
print("=" * 60)

agent = RAGMultiAgentSystem()

print("\n" + "=" * 60)
print("📚 KNOWLEDGE BASE CONTENTS")
print("=" * 60)
knowledge_items = [
    "Security Rule: All functions must handle user input validation",
    "Security Rule: Never use eval() or exec() on user input",
    "Performance Rule: Avoid recursion for large datasets (use iteration)",
    "Code Style Rule: Include docstrings for all functions",
    "Code Style Rule: Use snake_case for function names",
]
for i, item in enumerate(knowledge_items, 1):
    print(f"{i}. {item}")

print("\n" + "=" * 60)
print("🚀 RUNNING MULTI-AGENT SYSTEM WITH RAG")
print("=" * 60)

result = agent.process("Create a function to calculate factorial")

print("\n" + "=" * 60)
print("📊 FINAL ANALYSIS")
print("=" * 60)

if result and 'code' in result:
    code = result['code']
    print(f"\n📝 Code length: {len(code)} characters")
    print(f"📊 Lines of code: {len(code.split('\n')) if code else 0}")

    if code:
        print(f"\n💻 Generated Code:\n{code}")

        print("\n🔍 RAG Rule Compliance Check:")
        if "def factorial" in code or "def calculate_factorial" in code:
            print("✅ Has function definition")
        if "docstring" in code.lower() or '"""' in code:
            print("✅ Has docstring (follows RAG rule)")
        else:
            print("⚠️ No docstring found")
        if "snake_case" in code.lower():
            print("✅ Uses snake_case (follows RAG rule)")
        else:
            print("⚠️ Couldn't verify snake_case naming")
    else:
        print("⚠️ No code was generated.")
else:
    print("⚠️ No result returned from process()")

print("\n✅ Демонстрация завершена!")

🔄 Загрузка модели Qwen из Hugging Face...


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!
✅ Created new knowledge_base collection
✅ Added 5 documents to RAG knowledge base
🚀 Air-gapped multi-agent system with RAG ready!
📚 Knowledge base contains 5 rules
✅ Using Qwen from Hugging Face for both planning and coding

📚 KNOWLEDGE BASE CONTENTS
1. Security Rule: All functions must handle user input validation
2. Security Rule: Never use eval() or exec() on user input
3. Performance Rule: Avoid recursion for large datasets (use iteration)
4. Code Style Rule: Include docstrings for all functions
5. Code Style Rule: Use snake_case for function names

🚀 RUNNING MULTI-AGENT SYSTEM WITH RAG

📋 Task: Create a function to calculate factorial
--------------------------------------------------
🧠 Qwen is planning...

📝 Plan:
Plan:

Step 1: Define the function
- Function name: factorial
- Function parameters: None
- Function return type: integer

Step 2: Validate user input
- Ask the user for input
- Validate the input to ensure it is a positive integer
- If the inp

In [4]:
# ============================================================
# 🚀 ASYNCHRONOUS RL TRAINING MODULE FOR REASONING TASKS
# ============================================================
# This module extends VMAR with async RL capabilities:
# - Asynchronous rollout and training workers
# - Off-policy learning with importance sampling
# - vLLM integration for fast inference
# - Gradual task difficulty progression (curriculum learning)
# ============================================================

import asyncio
import multiprocessing as mp
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple
from collections import deque
import random
import json
import time
import numpy as np
from datetime import datetime
import threading
from concurrent.futures import ThreadPoolExecutor

# ============================================================
# CONFIGURATION
# ============================================================

@dataclass
class AsyncRLConfig:
    """Configuration for asynchronous RL training"""
    # Rollout settings
    num_rollout_workers: int = 2
    num_train_workers: int = 1
    batch_size: int = 32
    rollout_steps: int = 128

    # Experience buffer
    buffer_size: int = 10000
    importance_sampling: bool = True

    # Training settings
    learning_rate: float = 1e-5
    gradient_clip: float = 1.0
    train_interval: int = 4  # Train every N rollouts

    # Model settings
    model_name: str = "qwen2.5:1.5b"
    use_vllm: bool = True

    # Reward settings
    reward_type: str = "exact_match"  # or "llm_judge"

    # Curriculum learning
    curriculum: bool = True
    difficulty_levels: List[str] = None

    # Off-policy settings
    use_importance_sampling: bool = True
    v_trace_lambda: float = 0.95

    def __post_init__(self):
        if self.difficulty_levels is None:
            self.difficulty_levels = [
                "easy",  # Simple math (2+2)
                "medium",  # Two-step reasoning
                "hard"  # Complex chain-of-thought
            ]


# ============================================================
# EXPERIENCE BUFFER WITH IMPORTANCE SAMPLING
# ============================================================

class ExperienceBuffer:
    """
    Off-policy experience buffer with importance sampling.
    Supports V-trace and weighted importance sampling.
    """

    def __init__(self, max_size: int = 10000):
        self.buffer = deque(maxlen=max_size)
        self.max_size = max_size

    def add(self, experience):
        """Add a single experience to the buffer"""
        self.buffer.append(experience)

    def add_batch(self, experiences):
        """Add a batch of experiences"""
        for exp in experiences:
            self.buffer.append(exp)

    def sample(self, batch_size: int):
        """Sample a random batch from the buffer"""
        if len(self.buffer) < batch_size:
            return None
        return random.sample(self.buffer, batch_size)

    def sample_with_importance(self, batch_size: int, current_policy_probs: Dict = None):
        """
        Sample with importance sampling weights.
        Returns experiences with IS weights.
        """
        if len(self.buffer) < batch_size:
            return None

        # Simple uniform sampling with weights for now
        # In production, use behavior policy vs target policy
        batch = random.sample(self.buffer, batch_size)

        # Add importance weights (1.0 for now - assume same policy)
        for exp in batch:
            exp['is_weight'] = 1.0

        return batch

    def __len__(self):
        return len(self.buffer)


# ============================================================
# REWARD FUNCTION
# ============================================================

class RewardFunction:
    """
    Reward calculation for reasoning tasks.
    Supports exact match, partial match, and LLM-based evaluation.
    """

    def __init__(self, reward_type: str = "exact_match"):
        self.reward_type = reward_type

    def compute_reward(self, prediction: str, ground_truth: str) -> float:
        """
        Compute reward based on prediction vs ground truth.
        """
        if self.reward_type == "exact_match":
            return 1.0 if prediction.strip() == ground_truth.strip() else 0.0

        elif self.reward_type == "partial":
            # Partial credit: count overlapping tokens
            pred_tokens = set(prediction.lower().split())
            gt_tokens = set(ground_truth.lower().split())
            if not gt_tokens:
                return 0.0
            overlap = len(pred_tokens & gt_tokens) / len(gt_tokens)
            return min(overlap, 1.0)

        elif self.reward_type == "llm_judge":
            # Would call an LLM to judge reasoning quality
            # For now, fallback to partial match
            return self.compute_reward(prediction, ground_truth)

        return 0.0


# ============================================================
# TASK GENERATOR WITH CURRICULUM LEARNING
# ============================================================

class TaskGenerator:
    """
    Generates reasoning tasks with increasing difficulty.
    Implements curriculum learning.
    """

    def __init__(self, difficulty_levels: List[str] = None):
        if difficulty_levels is None:
            difficulty_levels = ["easy", "medium", "hard"]
        self.difficulty_levels = difficulty_levels
        self.current_level = 0  # Start with easiest
        self.performance_history = deque(maxlen=20)
        self.mastery_threshold = 0.8

    def generate_task(self) -> Tuple[str, str, str]:
        """
        Generate a task with question, answer, and difficulty.
        Returns: (task_text, answer, difficulty)
        """
        difficulty = self.difficulty_levels[self.current_level]

        if difficulty == "easy":
            return self._generate_easy()
        elif difficulty == "medium":
            return self._generate_medium()
        else:
            return self._generate_hard()

    def _generate_easy(self) -> Tuple[str, str, str]:
        """Simple arithmetic"""
        a = random.randint(1, 10)
        b = random.randint(1, 10)
        op = random.choice(['+', '-', '*'])
        if op == '+':
            answer = str(a + b)
        elif op == '-':
            answer = str(a - b)
        else:
            answer = str(a * b)
        task = f"Calculate: {a} {op} {b} = ?"
        return task, answer, "easy"

    def _generate_medium(self) -> Tuple[str, str, str]:
        """Word problems / two-step reasoning"""
        # Simple word problem
        items = ["apples", "oranges", "bananas"]
        item = random.choice(items)
        count = random.randint(2, 5)
        action = random.choice(["gave away", "bought", "sold"])
        if action == "bought":
            answer = f"{count} {item}"
        else:
            answer = f"{count} {item}"
        task = f"If you have {count} {item} and you {action} them, how many do you have?"
        return task, answer, "medium"

    def _generate_hard(self) -> Tuple[str, str, str]:
        """Complex reasoning / multi-step"""
        a = random.randint(1, 20)
        b = random.randint(1, 10)
        answer = str(a * b + b)
        task = f"If a box contains {a} apples and each apple has {b} seeds, how many seeds in total if you add {b} more apples?"
        return task, answer, "hard"

    def update_difficulty(self, success_rate: float):
        """
        Update difficulty based on success rate.
        """
        self.performance_history.append(success_rate)

        # Check if we should advance to next difficulty
        if len(self.performance_history) >= 5:
            recent_avg = sum(self.performance_history) / len(self.performance_history)
            if recent_avg > self.mastery_threshold:
                self.current_level = min(self.current_level + 1, len(self.difficulty_levels) - 1)
                print(f"📈 Advancing to difficulty: {self.difficulty_levels[self.current_level]}")
            elif recent_avg < 0.3 and self.current_level > 0:
                self.current_level -= 1
                print(f"📉 Decreasing to difficulty: {self.difficulty_levels[self.current_level]}")


# ============================================================
# VLLM INFERENCE WRAPPER
# ============================================================

# ============================================================
# VLLM INFERENCE WRAPPER (FULLY FIXED)
# ============================================================

class VLLMInference:
    """
    Fast inference using vLLM (or fallback to Ollama).
    Provides batching and low-latency generation.
    """

    def __init__(self, model_name: str = "qwen2.5:1.5b", use_vllm: bool = True):
        self.model_name = model_name
        self.use_vllm = use_vllm
        self.max_batch_size = 8

        if self.use_vllm:
            try:
                # Try to import vLLM
                from vllm import LLM, SamplingParams
                self.llm = LLM(
                    model=model_name,
                    tensor_parallel_size=1,
                    max_num_batched_tokens=8192
                )
                print(f"✅ vLLM loaded with model: {model_name}")
            except ImportError:
                print("⚠️ vLLM not available, falling back to Ollama")
                self.use_vllm = False
            except Exception as e:
                print(f"⚠️ vLLM initialization error: {e}, falling back to Ollama")
                self.use_vllm = False

        if not self.use_vllm:
            # Use the OllamaAPI class from the notebook
            try:
                self.ollama = OllamaAPI()
                print(f"✅ Using Ollama with model: {model_name}")
            except NameError:
                # Fallback: simple requests-based implementation
                import requests
                class SimpleOllama:
                    def __init__(self, host='http://127.0.0.1:11434'):
                        self.host = host
                        self.api_url = f"{host}/api/generate"

                    def generate(self, model, prompt, options=None):
                        payload = {
                            "model": model,
                            "prompt": prompt,
                            "stream": False,
                            "options": options or {}
                        }
                        try:
                            response = requests.post(self.api_url, json=payload, timeout=180)
                            if response.status_code == 200:
                                return response.json().get('response', '')
                            return f"ERROR: {response.status_code}"
                        except Exception as e:
                            return f"ERROR: {e}"
                self.ollama = SimpleOllama()
                print(f"⚠️ Using simple Ollama fallback with model: {model_name}")

    def generate(self, prompts: List[str], temperature: float = 0.3) -> List[str]:
        """
        Generate responses for a list of prompts.
        """
        if not prompts:
            return []

        if self.use_vllm:
            try:
                from vllm import SamplingParams
                sampling_params = SamplingParams(
                    temperature=temperature,
                    max_tokens=512,
                    top_p=0.9
                )
                outputs = self.llm.generate(prompts, sampling_params)
                return [out.outputs[0].text.strip() for out in outputs]
            except Exception as e:
                print(f"⚠️ vLLM error: {e}, falling back to Ollama")
                self.use_vllm = False

        # Ollama fallback
        results = []
        for prompt in prompts:
            response = self.ollama.generate(self.model_name, prompt, {
                "temperature": temperature,
                "num_predict": 512
            })
            results.append(response.strip())
        return results


# ============================================================
# ASYNCHRONOUS RL TRAINER
# ============================================================

class AsyncRLTrainer:
    """
    Asynchronous RL trainer for reasoning tasks.
    Implements parallel rollout and training workers.
    """

    def __init__(self, config: AsyncRLConfig = None):
        self.config = config or AsyncRLConfig()

        # Initialize components
        self.inference = VLLMInference(
            self.config.model_name,
            use_vllm=self.config.use_vllm
        )
        self.buffer = ExperienceBuffer(self.config.buffer_size)
        self.reward_fn = RewardFunction(self.config.reward_type)
        self.task_gen = TaskGenerator(self.config.difficulty_levels)

        # Statistics
        self.stats = {
            "total_rollouts": 0,
            "total_training_steps": 0,
            "average_reward": 0.0,
            "success_rate": 0.0,
            "difficulty": "easy"
        }

        # Running state
        self.running = False
        self.rollout_threads = []
        self.train_thread = None

        print(f"🚀 AsyncRLTrainer initialized with model: {self.config.model_name}")
        print(f"   Workers: {self.config.num_rollout_workers} rollout, {self.config.num_train_workers} train")
        print(f"   Curriculum: {self.config.curriculum}")
        print(f"   Off-policy IS: {self.config.use_importance_sampling}")

    def _rollout_worker(self, worker_id: int):
        """
        Worker that generates experiences through interaction.
        Runs continuously in background.
        """
        print(f"🔄 Rollout worker {worker_id} started")

        while self.running:
            try:
                # Generate task
                task, answer, difficulty = self.task_gen.generate_task()

                # Generate response using current model
                prompt = self._build_prompt(task)
                responses = self.inference.generate([prompt], temperature=0.3)

                if responses:
                    response = responses[0]
                    # Compute reward
                    reward = self.reward_fn.compute_reward(response, answer)

                    # Store experience
                    experience = {
                        'task': task,
                        'prompt': prompt,
                        'response': response,
                        'answer': answer,
                        'reward': reward,
                        'difficulty': difficulty,
                        'worker_id': worker_id,
                        'timestamp': time.time()
                    }
                    self.buffer.add(experience)
                    self.stats["total_rollouts"] += 1

                    # Update curriculum
                    if self.config.curriculum and len(self.buffer) > 10:
                        recent_rewards = [exp['reward'] for exp in list(self.buffer.buffer)[-20:]]
                        if recent_rewards:
                            self.task_gen.update_difficulty(sum(recent_rewards) / len(recent_rewards))
                            self.stats["difficulty"] = self.task_gen.difficulty_levels[self.task_gen.current_level]

                    # Update statistics
                    self._update_stats()

                # Small delay to prevent resource hogging
                time.sleep(0.1)

            except Exception as e:
                print(f"⚠️ Rollout worker {worker_id} error: {e}")
                time.sleep(1.0)

        print(f"🔄 Rollout worker {worker_id} stopped")

    def _train_worker(self):
        """
        Worker that trains the model using experiences from buffer.
        Runs continuously and consumes experiences asynchronously.
        """
        print(f"🔄 Training worker started")

        while self.running:
            try:
                # Wait for enough experiences
                if len(self.buffer) < self.config.batch_size:
                    time.sleep(0.5)
                    continue

                # Sample experiences
                if self.config.use_importance_sampling:
                    batch = self.buffer.sample_with_importance(
                        self.config.batch_size,
                        current_policy_probs={}
                    )
                else:
                    batch = self.buffer.sample(self.config.batch_size)

                if batch is None:
                    time.sleep(0.2)
                    continue

                # Process batch - in real implementation, this would update model
                self._train_on_batch(batch)

                self.stats["total_training_steps"] += 1

                # Respect training interval
                if self.stats["total_training_steps"] % self.config.train_interval == 0:
                    time.sleep(0.1)

            except Exception as e:
                print(f"⚠️ Training worker error: {e}")
                time.sleep(1.0)

        print(f"🔄 Training worker stopped")

    def _build_prompt(self, task: str) -> str:
        """Build a prompt for the model"""
        return f"""Solve the following problem. Provide only the answer, no explanation.

Problem: {task}

Answer:"""

    def _train_on_batch(self, batch: List[Dict]):
        """
        Train on a batch of experiences.
        In real implementation, this would update model weights.
        """
        # Compute off-policy correction (importance sampling)
        if self.config.use_importance_sampling:
            is_weights = [exp.get('is_weight', 1.0) for exp in batch]
            # Adjust learning based on IS weights
            avg_weight = sum(is_weights) / len(is_weights)
            if avg_weight > 5.0:
                # Clamp to prevent instability
                is_weights = [min(w, 5.0) for w in is_weights]

        # In real implementation:
        # - Compute PPO/GRPO loss
        # - Apply gradient descent
        # - Update model

        # For demo: simulate training
        rewards = [exp['reward'] for exp in batch]
        avg_reward = sum(rewards) / len(rewards) if rewards else 0

        # Update moving average
        self.stats["average_reward"] = self.stats["average_reward"] * 0.99 + avg_reward * 0.01
        self.stats["success_rate"] = self.stats["success_rate"] * 0.98 + (avg_reward > 0.8) * 0.02

    def _update_stats(self):
        """Update statistics from recent experiences"""
        if len(self.buffer) > 0:
            recent = list(self.buffer.buffer)[-50:]
            if recent:
                rewards = [exp['reward'] for exp in recent]
                self.stats["average_reward"] = sum(rewards) / len(rewards)
                self.stats["success_rate"] = sum(1 for r in rewards if r > 0.8) / len(rewards)

    def start(self):
        """
        Start asynchronous training.
        """
        if self.running:
            print("⚠️ Already running")
            return

        self.running = True

        print("🚀 Starting asynchronous RL training...")
        print(f"   Model: {self.config.model_name}")
        print(f"   Rollout workers: {self.config.num_rollout_workers}")

        # Start rollout workers
        for i in range(self.config.num_rollout_workers):
            thread = threading.Thread(target=self._rollout_worker, args=(i,), daemon=True)
            thread.start()
            self.rollout_threads.append(thread)

        # Start training worker
        self.train_thread = threading.Thread(target=self._train_worker, daemon=True)
        self.train_thread.start()

        print("✅ All workers started")
        print("   Press Ctrl+C to stop")

    def stop(self):
        """
        Stop asynchronous training.
        """
        print("🛑 Stopping training...")
        self.running = False

        # Wait for threads to finish
        for thread in self.rollout_threads:
            thread.join(timeout=1.0)
        if self.train_thread:
            self.train_thread.join(timeout=1.0)

        print("✅ Training stopped")

    def get_stats(self) -> Dict:
        """
        Get current training statistics.
        """
        return {
            "total_rollouts": self.stats["total_rollouts"],
            "total_training_steps": self.stats["total_training_steps"],
            "average_reward": round(self.stats["average_reward"], 3),
            "success_rate": round(self.stats["success_rate"], 3),
            "buffer_size": len(self.buffer),
            "difficulty": self.stats["difficulty"]
        }


# ============================================================
# INTEGRATION WITH EXISTING VMAR SYSTEM
# ============================================================

class VMARAsyncTrainer:
    """
    Extended VMAR system with async RL capabilities.
    Combines verification logic with asynchronous training.
    """

    def __init__(self, base_vmar_system=None):
        self.base_system = base_vmar_system
        self.async_trainer = AsyncRLTrainer()
        self.verification_results = []

    def start_training(self):
        """Start async RL training with VMAR verification"""
        print("🚀 Starting VMAR + AsyncRL training")

        # Override the reward function to use VMAR verification
        self.async_trainer.reward_fn.reward_type = "vmar_verified"

        # Start training
        self.async_trainer.start()

        # Monitor verification
        self._monitor_training()

    def _monitor_training(self):
        """
        Monitor training and apply VMAR verification.
        """
        def monitor():
            while self.async_trainer.running:
                stats = self.async_trainer.get_stats()
                print(f"📊 Stats: {stats}")

                # Check for quality issues
                if stats["success_rate"] < 0.5 and stats["total_rollouts"] > 100:
                    print("⚠️ Success rate dropping, adjusting curriculum")
                    self.async_trainer.task_gen.current_level = max(
                        0, self.async_trainer.task_gen.current_level - 1
                    )

                time.sleep(5.0)

        # Run monitoring in background
        monitor_thread = threading.Thread(target=monitor, daemon=True)
        monitor_thread.start()

    def verify_and_train(self, task: str, ground_truth: str):
        """
        Verify a task using VMAR, then use results for training.
        """
        # Get VMAR verification result
        verified_result = self.base_system.process(task) if self.base_system else None

        # Generate experience
        prompt = self.async_trainer._build_prompt(task)
        response = self.async_trainer.inference.generate([prompt], temperature=0.3)[0]

        # Compute reward with verification
        if verified_result:
            reward = 1.0 if response == ground_truth else 0.0
        else:
            reward = self.async_trainer.reward_fn.compute_reward(response, ground_truth)

        # Add to buffer
        experience = {
            'task': task,
            'prompt': prompt,
            'response': response,
            'answer': ground_truth,
            'reward': reward,
            'verified': bool(verified_result)
        }
        self.async_trainer.buffer.add(experience)
        self.verification_results.append(experience)

        return response, reward, verified_result


# ============================================================
# DEMONSTRATION
# ============================================================

def demo_async_training():
    """
    Demonstrate the async RL training system.
    """
    print("=" * 60)
    print("🚀 ASYNC RL TRAINING DEMONSTRATION")
    print("=" * 60)

    # Create config
    config = AsyncRLConfig(
        num_rollout_workers=2,
        num_train_workers=1,
        batch_size=16,
        buffer_size=1000,
        use_vllm=True,
        curriculum=True
    )

    # Create trainer
    trainer = AsyncRLTrainer(config)

    # Start training
    trainer.start()

    # Run for a few seconds to show activity
    print("\n" + "=" * 60)
    print("📊 TRAINING PROGRESS")
    print("=" * 60)

    for i in range(10):
        time.sleep(1.0)
        stats = trainer.get_stats()
        print(f"[{i+1}s] Rollouts: {stats['total_rollouts']}, "
              f"Reward: {stats['average_reward']:.3f}, "
              f"Success: {stats['success_rate']:.3f}, "
              f"Difficulty: {stats['difficulty']}")

    # Stop training
    trainer.stop()

    # Show final stats
    print("\n" + "=" * 60)
    print("📊 FINAL STATISTICS")
    print("=" * 60)
    final_stats = trainer.get_stats()
    for key, value in final_stats.items():
        print(f"{key}: {value}")

    return trainer


# ============================================================
# COMPARISON: SYNCHRONOUS VS ASYNCHRONOUS
# ============================================================

# ============================================================
# COMPARISON: SYNCHRONOUS VS ASYNCHRONOUS
# ============================================================

def compare_sync_async():
    """
    Compare synchronous vs asynchronous performance.
    """
    print("=" * 60)
    print("📊 SYNCHRONOUS VS ASYNCHRONOUS COMPARISON")
    print("=" * 60)

    # Synchronous baseline
    print("\n🔹 Synchronous (batch-by-batch):")
    print("   - Generate → Train → Generate → Train")
    print("   - Worker idle waiting for training")
    print("   - Typical speedup: 1x")

    # Asynchronous
    print("\n🔹 Asynchronous (parallel):")
    print("   - Generate AND Train simultaneously")
    print("   - Rollout workers never idle")
    print("   - Typical speedup: 2-3x")

    # Off-policy
    print("\n🔹 Off-policy correction:")
    print("   - Can learn from older experiences")
    print("   - Importance sampling prevents instability")
    print("   - Replay buffer: 10000+ experiences")

    # Curriculum
    print("\n🔹 Curriculum learning:")
    print("   - Gradually increasing difficulty")
    print("   - Prevents model from plateauing")
    print("   - Adaptive based on performance")


# ============================================================
# RUN DEMONSTRATION
# ============================================================

if __name__ == "__main__":
    # Run the demo
    trainer = demo_async_training()

    # Show comparison
    compare_sync_async()

    print("\n✅ Async RL training module ready for integration with VMAR!")

🚀 ASYNC RL TRAINING DEMONSTRATION
⚠️ vLLM not available, falling back to Ollama
⚠️ Using simple Ollama fallback with model: qwen2.5:1.5b
🚀 AsyncRLTrainer initialized with model: qwen2.5:1.5b
   Workers: 2 rollout, 1 train
   Curriculum: True
   Off-policy IS: True
🚀 Starting asynchronous RL training...
   Model: qwen2.5:1.5b
   Rollout workers: 2
🔄 Rollout worker 0 started
🔄 Rollout worker 1 started
🔄 Training worker started
✅ All workers started
   Press Ctrl+C to stop

📊 TRAINING PROGRESS
[1s] Rollouts: 12, Reward: 0.000, Success: 0.000, Difficulty: easy
[2s] Rollouts: 32, Reward: 0.000, Success: 0.000, Difficulty: easy
[3s] Rollouts: 52, Reward: 0.000, Success: 0.000, Difficulty: easy
[4s] Rollouts: 71, Reward: 0.000, Success: 0.000, Difficulty: easy
[5s] Rollouts: 90, Reward: 0.000, Success: 0.000, Difficulty: easy
[6s] Rollouts: 109, Reward: 0.000, Success: 0.000, Difficulty: easy
[7s] Rollouts: 128, Reward: 0.000, Success: 0.000, Difficulty: easy
[8s] Rollouts: 148, Reward: 0.000

In [8]:
# ============================================================
# 🚀 ПОЛНАЯ СИСТЕМА: RAG + RL С ИСПОЛЬЗОВАНИЕМ HUGGING FACE
# ============================================================

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import random
import time
import numpy as np
from collections import deque
import threading

# ============================================================
# 1. ЗАГРУЗКА МОДЕЛИ HUGGING FACE
# ============================================================

print("🔄 Загрузка модели Qwen из Hugging Face...")
MODEL_NAME = "Qwen/Qwen2.5-1.5B"  # Или "Qwen/Qwen2.5-0.5B" для скорости

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print("✅ Модель загружена!")

def generate_response(prompt, max_new_tokens=512, temperature=0.7):
    """Генерация ответа через Hugging Face модель"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response[len(prompt):].strip()
    return response

# ============================================================
# 2. RAG MULTI-AGENT SYSTEM (АДАПТИРОВАН ДЛЯ HUGGING FACE)
# ============================================================

class RAGMultiAgentSystem:
    def __init__(self):
        # ✅ Модель Hugging Face (заменяет Ollama)
        self.model_name = MODEL_NAME
        self.tokenizer = tokenizer
        self.model = model

        # Симулируем атрибут ollama для совместимости с RL-частью
        # (но на самом деле используем generate_response)
        class HFAdapter:
            def __init__(self, parent):
                self.parent = parent
            def generate(self, prompt, options=None):
                max_tokens = options.get("num_predict", 800) if options else 800
                temp = options.get("temperature", 0.7) if options else 0.7
                return self.parent._generate(prompt, max_tokens, temp)

        self.ollama = HFAdapter(self)

        # Knowledge base
        self.knowledge_base = [
            "Security Rule: All functions must handle user input validation",
            "Security Rule: Never use eval() or exec() on user input",
            "Performance Rule: Avoid recursion for large datasets (use iteration)",
            "Code Style Rule: Include docstrings for all functions - use '''docstring'''",
            "Code Style Rule: Use snake_case for function names",
        ]

        # Простая имитация RAG (без chromadb для скорости в демо)
        self.rag = None  # Можно добавить, но для демо не обязательно
        print("🚀 Air-gapped multi-agent system with Hugging Face ready!")
        print("✅ Using Qwen for both planning and coding")

    def _generate(self, prompt, max_tokens=800, temperature=0.7):
        """Внутренний метод генерации"""
        return generate_response(prompt, max_tokens, temperature)

    def plan(self, task):
        print("🧠 Qwen is planning...")
        prompt = f"""Create a detailed, step-by-step plan for this task:

TASK: {task}

Provide the plan as clear, numbered steps. Focus on the algorithm and logic.
Be specific about what the code should do."""

        return self._generate(prompt, max_tokens=800, temperature=0.7)

    def code(self, plan):
        print("💻 Qwen is coding...")
        prompt = f"""Write Python code for this task:

{plan}

Rules to follow:
- Write clean, working Python code
- Include docstrings
- Use snake_case for function names
- Handle input validation
- Just write the code, no explanations

Code:"""

        response = self._generate(prompt, max_tokens=1500, temperature=0.3)
        return self._extract_code(response)

    def _extract_code(self, text):
        if not text:
            return ""
        if "```python" in text:
            code = text.split("```python")[1].split("```")[0]
            if code.strip():
                return code.strip()
        if "```" in text:
            code = text.split("```")[1].split("```")[0]
            if code.strip():
                return code.strip()
        return text.strip()

    def process(self, task):
        """Full multi-agent workflow"""
        print(f"\n📋 Task: {task}")
        print("-" * 50)

        plan = self.plan(task)
        print(f"\n📝 Plan:\n{plan}")
        print("-" * 50)

        code = self.code(plan)
        print(f"\n💻 Code:\n{code}")
        print("-" * 50)

        print("\n⚙️ Executing code...")
        if code.strip():
            try:
                safe_globals = {
                    "__builtins__": __builtins__,
                    "print": print,
                    "range": range,
                    "int": int,
                    "float": float,
                    "str": str,
                    "list": list,
                    "dict": dict,
                }
                exec(code, safe_globals)
                print("✅ Code executed successfully!")
            except Exception as e:
                print(f"❌ Code execution failed: {e}")
        else:
            print("⚠️ No code generated.")

        return {"plan": plan, "code": code}

# ============================================================
# 3. RL TRAINING LOOP (УПРОЩЁННАЯ ВЕРСИЯ ДЛЯ ДЕМО)
# ============================================================

class CompleteRLTrainingLoop:
    def __init__(self, model_wrapper, learning_rate=1e-5, gamma=0.99, eps_clip=0.2):
        self.model = model_wrapper
        self.gamma = gamma
        self.eps_clip = eps_clip
        self.buffer = deque(maxlen=10000)
        self.training_stats = {'total_updates': 0, 'avg_reward': 0.0}
        print("🔴 RL Training Loop initialized")

    def add_experience(self, experience):
        self.buffer.append(experience)

    def sample_batch(self, batch_size=32):
        if len(self.buffer) < batch_size:
            return None
        return random.sample(self.buffer, batch_size)

    def train_step(self, batch):
        if len(batch) < 8:
            return None

        rewards = [exp['reward'] for exp in batch]
        avg_reward = sum(rewards) / len(rewards)

        self.training_stats['total_updates'] += 1
        self.training_stats['avg_reward'] = self.training_stats['avg_reward'] * 0.9 + avg_reward * 0.1

        return {
            'total_loss': 0.1,
            'policy_loss': 0.05,
            'value_loss': 0.05,
            'avg_reward': avg_reward
        }

    def get_stats(self):
        return {
            'total_updates': self.training_stats['total_updates'],
            'avg_reward': self.training_stats['avg_reward'],
            'buffer_size': len(self.buffer)
        }

# ============================================================
# 4. INTEGRATED RL SYSTEM
# ============================================================

class IntegratedRLSystem:
    def __init__(self, base_system):
        self.base = base_system
        self.rl_loop = CompleteRLTrainingLoop(
            model_wrapper=base_system.ollama,
            learning_rate=1e-5,
            gamma=0.99,
            eps_clip=0.2
        )
        self.training_history = []
        print("✅ Integrated RL System initialized!")

    def train_on_task(self, task):
        print(f"\n📋 Training on: {task}")

        result = self.base.process(task)
        code = result.get('code', '')
        reward = min(len(code) / 100, 1.0) if code else 0.0

        experience = {
            'state': task,
            'action': code,
            'reward': reward,
            'done': True
        }

        self.rl_loop.add_experience(experience)
        self.training_history.append(experience)

        print(f"   Reward: {reward:.2f}")

        if len(self.rl_loop.buffer) >= 8:
            batch = self.rl_loop.sample_batch(8)
            if batch:
                loss_info = self.rl_loop.train_step(batch)
                if loss_info:
                    print(f"   🔴 RL Training: Avg Reward={loss_info['avg_reward']:.3f}")

        return experience

    def train_loop(self, tasks, num_epochs=1):
        print("=" * 60)
        print("🔴 RL TRAINING LOOP")
        print("=" * 60)

        for epoch in range(num_epochs):
            print(f"\n🔄 Epoch {epoch + 1}/{num_epochs}")
            for i, task in enumerate(tasks):
                self.train_on_task(task)
                if (i + 1) % 2 == 0:
                    stats = self.rl_loop.get_stats()
                    print(f"   📊 Stats: Updates={stats['total_updates']}, "
                          f"Avg Reward={stats['avg_reward']:.3f}, "
                          f"Buffer={stats['buffer_size']}")

# ============================================================
# 5. ЗАПУСК ДЕМОНСТРАЦИИ
# ============================================================

print("\n" + "=" * 60)
print("🚀 ЗАПУСК RAG + RL СИСТЕМЫ")
print("=" * 60)

# Создаём базовую систему
base_system = RAGMultiAgentSystem()

# Создаём систему с RL
rl_system = IntegratedRLSystem(base_system)

# Задачи для обучения
tasks = [
    "Create a function to calculate factorial",
    "Create a function to reverse a string",
    "Create a function to check if a number is prime",
    "Create a function to calculate Fibonacci numbers",
]

# Запускаем обучение
rl_system.train_loop(tasks, num_epochs=1)

# Показываем результат
print("\n" + "=" * 60)
print("📊 РЕЗУЛЬТАТЫ ОБУЧЕНИЯ")
print("=" * 60)
stats = rl_system.rl_loop.get_stats()
print(f"Total updates: {stats['total_updates']}")
print(f"Average reward: {stats['avg_reward']:.3f}")
print(f"Buffer size: {stats['buffer_size']}")
print("\n✅ Демонстрация завершена!")

🔄 Загрузка модели Qwen из Hugging Face...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Модель загружена!

🚀 ЗАПУСК RAG + RL СИСТЕМЫ
🚀 Air-gapped multi-agent system with Hugging Face ready!
✅ Using Qwen for both planning and coding
🔴 RL Training Loop initialized
✅ Integrated RL System initialized!
🔴 RL TRAINING LOOP

🔄 Epoch 1/1

📋 Training on: Create a function to calculate factorial

📋 Task: Create a function to calculate factorial
--------------------------------------------------
🧠 Qwen is planning...

📝 Plan:
The function should take a non-negative integer as input and return the factorial of that number.

1. Check if the input number is negative. If it is, return an error message since the factorial is not defined for negative numbers.

2. If the input number is 0 or 1, return 1 since the factorial of 0 and 1 is 1.

3. Create a loop to iterate from 2 to the input number.

4. In each iteration, multiply the current number by the previous number in the loop.

5. Return the final result after the loop completes.

6. If the input number is not a non-negative integer, 